# Inference LLM on notebooks

In [7]:
!pip -q install pyngrok

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import torch

In [4]:
from huggingface_hub import login

login(token="REMOVED_HF_TOKEN")

In [17]:
NGROK_AUTH_TOKEN = "REMOVED_NGROK_TOKEN"

In [5]:
model_name = "minn4/text2diagram-Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [11]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from pyngrok import ngrok

app = FastAPI()

class Request(BaseModel):
    prompt: str

@app.post("/generate")
def generate(req: Request):

    messages = [
        {"role": "user", "content": req.prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=128)

    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return {"response": text}

public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)

server = uvicorn.Server(
    uvicorn.Config(app, host="localhost", port=8000)
)

await server.serve()

PUBLIC URL: NgrokTunnel: "https://victoria-communicable-sometimes.ngrok-free.dev" -> "http://localhost:8000"


INFO:     Started server process [5337]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8000 (Press CTRL+C to quit)


INFO:     ::1:54486 - "POST / HTTP/1.1" 404 Not Found
